In [2]:
import gcsfs
import os

import dask
import s3fs
import xarray as xr
from distributed import Client

# --- file destination
dst = 's3://carbonplan-carbon-removal/era5/preprocessed_zarr/'
store_region = 'us'


In [3]:
client = Client()
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://cluster-aprjn.dask.host/jupyter/proxy/8787/status,
Dashboard: https://cluster-aprjn.dask.host/jupyter/proxy/8787/status,Workers: 8
Total threads: 32,Total memory: 122.05 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:44253,Workers: 0
Dashboard: https://cluster-aprjn.dask.host/jupyter/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:36567,Total threads: 4
Dashboard: https://cluster-aprjn.dask.host/jupyter/proxy/35633/status,Memory: 15.26 GiB
Nanny: tcp://127.0.0.1:44237,


In [4]:
# --- 
gcp_path = 'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3/'
ar_full_37_1h = xr.open_zarr(gcp_path)

# # [ convert longitude to -180 to 180 ]
# ar_full_37_1h['longitude'] = ((ar_full_37_1h['longitude'] + 180) % 360) - 180
# ar_full_37_1h = ar_full_37_1h.sortby('longitude').chunk({'time': 500, 'latitude': 100, 'longitude': 100})

ar_full_37_1h

<xarray.Dataset> Size: 4PB
Dimensions:                                                          (
                                                                      time: 1323648,
                                                                      latitude: 721,
                                                                      longitude: 1440,
                                                                      level: 37)
Coordinates:
  * latitude                                                         (latitude) float32 3kB ...
  * level                                                            (level) int64 296B ...
  * longitude                                                        (longitude) float32 6kB ...
  * time                                                             (time) datetime64[ns] 11MB ...
Data variables: (12/273)
    100m_u_component_of_wind                                         (time, latitude, longitude) float32 5TB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    100m_v_component_of_wind                                         (time, latitude, longitude) float32 5TB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    10m_u_component_of_neutral_wind                                  (time, latitude, longitude) float32 5TB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    10m_u_component_of_wind                                          (time, latitude, longitude) float32 5TB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    10m_v_component_of_neutral_wind                                  (time, latitude, longitude) float32 5TB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    10m_v_component_of_wind                                          (time, latitude, longitude) float32 5TB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    ...                                                               ...
    wave_spectral_directional_width_for_swell                        (time, latitude, longitude) float32 5TB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    wave_spectral_directional_width_for_wind_waves                   (time, latitude, longitude) float32 5TB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    wave_spectral_kurtosis                                           (time, latitude, longitude) float32 5TB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    wave_spectral_peakedness                                         (time, latitude, longitude) float32 5TB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    wave_spectral_skewness                                           (time, latitude, longitude) float32 5TB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    zero_degree_level                                                (time, latitude, longitude) float32 5TB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
Attributes:
    last_updated:           2025-07-23 01:54:31.932125+00:00
    valid_time_start:       1940-01-01
    valid_time_stop:        2025-04-30
    valid_time_stop_era5t:  2025-07-17

In [5]:

# data selection
varlist = ["2m_temperature",
        "skin_reservoir_content",
        "volumetric_soil_water_layer_1",
        "volumetric_soil_water_layer_2",
        "volumetric_soil_water_layer_3",
        "volumetric_soil_water_layer_4",
        "soil_temperature_level_1",
        "soil_temperature_level_2",
        "soil_temperature_level_3",
        "soil_temperature_level_4",
        "potential_evaporation",
        "runoff",
        "surface_runoff",
        "sub_surface_runoff",
        "evaporation",
        "total_precipitation",
        "geopotential",
        "land_sea_mask",
        "soil_type"
        ]
minlat, maxlat = 24, 50
minlon, maxlon = -125, -65
mintime, maxtime = '2000', '2020'
# +++++++++++++++++++++++++++++++++++++++++++++++++

dsx = ar_full_37_1h[varlist].sel(latitude=slice(maxlat, minlat), 
                                longitude=slice(360+minlon, 360+maxlon),
                                time=slice(mintime, maxtime),
                           level = 1000,
                            )

dsx.chunk({'time': -1, 'latitude': 12, 'longitude':12})


dsx1 = dsx.isel(time=slice(0,1000))



In [8]:
dsx

<xarray.Dataset> Size: 354GB
Dimensions:                        (time: 184104, latitude: 105, longitude: 241)
Coordinates:
  * latitude                       (latitude) float32 420B 50.0 49.75 ... 24.0
    level                          int64 8B 1000
  * longitude                      (longitude) float32 964B 235.0 ... 295.0
  * time                           (time) datetime64[ns] 1MB 2000-01-01 ... 2...
Data variables: (12/19)
    2m_temperature                 (time, latitude, longitude) float32 19GB dask.array<chunksize=(1, 105, 241), meta=np.ndarray>
    skin_reservoir_content         (time, latitude, longitude) float32 19GB dask.array<chunksize=(1, 105, 241), meta=np.ndarray>
    volumetric_soil_water_layer_1  (time, latitude, longitude) float32 19GB dask.array<chunksize=(1, 105, 241), meta=np.ndarray>
    volumetric_soil_water_layer_2  (time, latitude, longitude) float32 19GB dask.array<chunksize=(1, 105, 241), meta=np.ndarray>
    volumetric_soil_water_layer_3  (time, latitude, longitude) float32 19GB dask.array<chunksize=(1, 105, 241), meta=np.ndarray>
    volumetric_soil_water_layer_4  (time, latitude, longitude) float32 19GB dask.array<chunksize=(1, 105, 241), meta=np.ndarray>
    ...                             ...
    sub_surface_runoff             (time, latitude, longitude) float32 19GB dask.array<chunksize=(1, 105, 241), meta=np.ndarray>
    evaporation                    (time, latitude, longitude) float32 19GB dask.array<chunksize=(1, 105, 241), meta=np.ndarray>
    total_precipitation            (time, latitude, longitude) float32 19GB dask.array<chunksize=(1, 105, 241), meta=np.ndarray>
    geopotential                   (time, latitude, longitude) float32 19GB dask.array<chunksize=(1, 105, 241), meta=np.ndarray>
    land_sea_mask                  (time, latitude, longitude) float32 19GB dask.array<chunksize=(1, 105, 241), meta=np.ndarray>
    soil_type                      (time, latitude, longitude) float32 19GB dask.array<chunksize=(1, 105, 241), meta=np.ndarray>
Attributes:
    last_updated:           2025-07-23 01:54:31.932125+00:00
    valid_time_start:       1940-01-01
    valid_time_stop:        2025-04-30
    valid_time_stop_era5t:  2025-07-17

In [7]:
dsx1.drop_encoding().to_zarr('s3://carbonplan-carbon-removal/era5/preprocessed_zarr/test1.zarr',
           mode='w')

/opt/coiled/env/lib/python3.13/site-packages/zarr/api/asynchronous.py:229: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/opt/coiled/env/lib/python3.13/site-packages/distributed/client.py:3371: UserWarning: Sending large graph of size 504.03 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0xe13315045450>
Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0xe13314ec5070>, 528.586855002)])']
connector: <aiohttp.connector.TCPConnector object at 0xe13315044e10>


In [ ]:
# ---